In [8]:
# Have to use my_gbd_sunset environment
import os
# Shared Functions
from db_queries import get_location_metadata, get_age_metadata, get_population, get_cause_metadata, get_outputs, get_covariate_estimates
from get_draws.api import get_draws
import pandas as pd

In [9]:
import sys
from pathlib import Path
from datetime import date
import json

sys.path.insert(0, '/ihme/homes/bcreiner/repos/idd-forecast-mbp/src')
from idd_forecast_mbp import constants as mbpc


In [10]:
data_date = mbpc.GBD_DATA_DATE
data_path = mbpc.GBD_DATA_PATH / data_date
data_path.mkdir(parents=True, exist_ok=True)
with open(data_path / "gbd_constants.json", "w") as f:
    json.dump(mbpc.gbd_constants, f, indent=2)

make_current = True
if make_current:
    current_link = mbpc.GBD_DATA_PATH / "current"
    if current_link.is_symlink() or current_link.exists():
        current_link.unlink()
    current_link.symlink_to(data_date)

In [11]:
sex_ids = [1,2,3]
# Age metadata
age_metadata = get_age_metadata(release_id = mbpc.gbd_constants['release_2023_id'])
age_metadata = age_metadata[["age_group_id", "age_group_years_start", "age_group_years_end", "age_group_name"]]
# new_rows <- data.frame(age_group_id = c(1,22),
#                        age_group_years_start = c(0, 0),
#                        age_group_years_end = c(5, 125),
#                        age_group_name = c("Under 5", "All age"))
age_metadata = pd.concat([age_metadata, pd.DataFrame({"age_group_id": [1, 22, 27],
                                                      "age_group_years_start": [0, 0, 0],
                                                      "age_group_years_end": [5, 125, 125],
                                                      "age_group_name": ["Under 5", "All age", "Age standardized"]})], ignore_index=True)
age_metadata.to_parquet(data_path / "age_metadata.parquet", index=False)

In [12]:
# Get hierarchy
fhs_hierarchy_2023 = get_location_metadata(location_set_id = mbpc.gbd_constants['fhs_location_set_id'], release_id = mbpc.gbd_constants['release_2023_id'])
gbd_hierarchy_2023 = get_location_metadata(location_set_id = mbpc.gbd_constants['gbd_location_set_id'], release_id = mbpc.gbd_constants['release_2023_id'])
cols_to_drop = ["start_date", "end_date", "date_inserted", "last_updated", "last_updated_by", "last_updated_action"]
fhs_hierarchy_2023 = fhs_hierarchy_2023.drop(columns=cols_to_drop)
gbd_hierarchy_2023 = gbd_hierarchy_2023.drop(columns=cols_to_drop)

fhs_hierarchy_2023.to_parquet(data_path / "fhs_2023_modeling_hierarchy.parquet", index=False)
gbd_hierarchy_2023.to_parquet(data_path / "gbd_2023_modeling_hierarchy.parquet", index=False)

location_ids = gbd_hierarchy_2023[gbd_hierarchy_2023['level'] <= 4]['location_id'].tolist()

years = list(range(2000, 2024))

In [13]:

age_group_ids = age_metadata['age_group_id'].tolist()
# Get population
gbd_population = get_population(
    age_group_id=age_group_ids,
    release_id=mbpc.gbd_constants['release_2023_id'],
    year_id=years,
    location_id=location_ids,
    sex_id=sex_ids
)


fhs_population = get_population(
    age_group_id=age_group_ids,
    release_id=mbpc.gbd_constants['release_2023_id'],
    year_id=years,
    location_id=fhs_hierarchy_2023['location_id'].tolist(),
    sex_id=sex_ids
)

gbd_population = gbd_population[["age_group_id", "location_id", "year_id", "sex_id", "population"]].copy()
fhs_population = fhs_population[["age_group_id", "location_id", "year_id", "sex_id", "population"]].copy()

gbd_population.to_parquet(data_path / "gbd_2023_population.parquet", index=False)
fhs_population.to_parquet(data_path / "fhs_2023_population.parquet", index=False)

In [14]:
for cause_key in ['malaria', 'dengue']:
    cause_info = mbpc.cause_map[cause_key]
    print(f"Processing cause: {cause_info['cause_name']}")
    cause_id = cause_info['cause_id']
    cause_name = cause_info['cause_name']
    fhs_cause_name = cause_info['fhs_cause_name']

    print(f"Getting AA results for cause: {cause_name}")
    aa_results = get_outputs(
        "cause",
        cause_id=cause_id,
        measure_id=[1,2,3,4,5,6],
        year_id=years,
        location_id=location_ids,
        age_group_id=[22],
        release_id=mbpc.gbd_constants['release_2023_id'],
        metric_id=[1, 3],
        sex_id=[3],
        compare_version_id=mbpc.gbd_constants['compare_2023_v']
    )
    aa_results = aa_results.merge(gbd_hierarchy_2023, on="location_id", how="left")
    aa_results = aa_results.merge(gbd_population, on=["age_group_id", "location_id", "year_id", "sex_id"], how="left")
    aa_results.to_parquet(f"{data_path}/aa_{cause_key}_results.parquet", index=False)

    print(f"Getting AS results for cause: {cause_name}")
    as_results = get_outputs(
        "cause",
        cause_id=cause_id,
        measure_id=[1,6],
        year_id=years,
        location_id=location_ids,
        age_group_id=age_group_ids,
        release_id=mbpc.gbd_constants['release_2023_id'],
        metric_id=[1, 3],
        sex_id=[1, 2, 3],
        compare_version_id=mbpc.gbd_constants['compare_2023_v']
    )

    as_results = as_results.merge(gbd_hierarchy_2023, on="location_id", how="left")
    as_results = as_results.merge(gbd_population, on=["age_group_id", "location_id", "year_id", "sex_id"], how="left")
    as_results.to_parquet(f"{data_path}/as_{cause_key}_results.parquet", index=False)

Processing cause: Malaria
Getting AA results for cause: Malaria
Getting AS results for cause: Malaria
Processing cause: Dengue
Getting AA results for cause: Dengue
Getting AS results for cause: Dengue
